# Notebook 1: Experiment 1 — Same-Stock Prediction (80/20)
## A Comparative Analysis of BiLSTM and BiGRU for Stock Price Prediction

**Experiment:** Train on each stock's daily data and predict its own future prices.  
**Train/Test Split:** 80/20 (chronological)  
**Models:** BiLSTM, BiGRU, LSTM, GRU  
**Stocks:** TLKM, BBCA, ASII, UNVR  
**Metrics:** MSE, RMSE, MAE, MAPE, R² Score  


In [5]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from stock_prediction_utils import *

set_seed()
set_ieee_style()
check_gpu()

DATA_DIR = 'dataset'

TRAIN_RATIO = 0.8
RATIO_LABEL = '80_20'
EXP_LABEL = f'Exp1_{RATIO_LABEL}'

os.makedirs(f'figures/{EXP_LABEL}', exist_ok=True)
os.makedirs(f'models/{EXP_LABEL}', exist_ok=True)
os.makedirs('results', exist_ok=True)

print(f"Experiment 1 - Same Stock Prediction (80/20)")
print(f"Train ratio: {TRAIN_RATIO}, Test ratio: {1-TRAIN_RATIO}")



Device Configuration:
  GPUs available: 1
  CPUs available: 1
  TensorFlow will use GPU for computations
Experiment 1 - Same Stock Prediction (80/20)
Train ratio: 0.8, Test ratio: 0.19999999999999996


In [6]:
# Load all daily data
print("Loading daily data...")
daily_data = load_all_daily_data(DATA_DIR)
print("\nAll daily data loaded!")


Loading daily data...
  TLKM: 5243 records, Date range: 2004-09-28 to 2025-12-31
  BBCA: 5244 records, Date range: 2004-09-28 to 2025-12-31
  ASII: 5244 records, Date range: 2004-09-28 to 2025-12-31
  UNVR: 5245 records, Date range: 2004-09-28 to 2025-12-31

All daily data loaded!


In [7]:
# Reload the module to get the latest changes
import importlib
import stock_prediction_utils
importlib.reload(stock_prediction_utils)
from stock_prediction_utils import *
print("✓ Module reloaded successfully")

stock_prediction_utils.py loaded successfully!
  ProportionScaler max value: 10501.0
  Lookback: 1, Epochs: 100, Batch size: 64
  Architecture: 2 layers, 64 units, dropout=0.2
  Stocks: ['TLKM', 'BBCA', 'ASII', 'UNVR']
  Models: ['BiLSTM', 'BiGRU', 'LSTM', 'GRU']

GPU SETUP - CUDA Available
Number of GPUs detected: 1
  GPU 0: /physical_device:GPU:0

Memory growth enabled (dynamic allocation)
TensorFlow configured to use GPU


Device Configuration:
  GPUs available: 1
  CPUs available: 1
  TensorFlow will use GPU for computations
✓ Module reloaded successfully


In [8]:
# ============================================================
# INTERACTIVE DATA SPLIT VISUALIZATION
# ============================================================

# Individual stock data split visualizations
print("Generating interactive data split visualizations...\n")

for stock in STOCKS:
    fig, html_file = create_interactive_data_split_visualization(
        daily_data[stock], 
        train_ratio=TRAIN_RATIO,
        stock_name=stock,
        experiment_label=EXP_LABEL,
        save_dir=f'figures/{EXP_LABEL}'
    )
    print(f"✓ {stock} split visualization saved: {html_file}")
    fig.show()

print("\n")

# Summary visualization - all stocks
print("Generating summary data split visualization for all stocks...\n")
fig_summary, html_summary = create_interactive_split_summary_visualization(
    daily_data,
    train_ratio=TRAIN_RATIO,
    experiment_label=EXP_LABEL,
    save_dir=f'figures/{EXP_LABEL}'
)
print(f"✓ Summary visualization saved: {html_summary}")
fig_summary.show()

print("\n")

# Statistics bar chart
print("Generating data split statistics bar chart...\n")
fig_stats, html_stats = create_interactive_split_bar_chart(
    daily_data,
    train_ratio=TRAIN_RATIO,
    experiment_label=EXP_LABEL,
    save_dir=f'figures/{EXP_LABEL}'
)
print(f"✓ Statistics chart saved: {html_stats}")
fig_stats.show()

print("\n✓ All data split visualizations generated successfully!")

Generating interactive data split visualizations...

✓ TLKM split visualization saved: figures/Exp1_80_20/interactive_data_split_TLKM_80_19.html


✓ BBCA split visualization saved: figures/Exp1_80_20/interactive_data_split_BBCA_80_19.html


✓ ASII split visualization saved: figures/Exp1_80_20/interactive_data_split_ASII_80_19.html


✓ UNVR split visualization saved: figures/Exp1_80_20/interactive_data_split_UNVR_80_19.html




Generating summary data split visualization for all stocks...

✓ Summary visualization saved: figures/Exp1_80_20/interactive_data_split_all_stocks_80_19.html




Generating data split statistics bar chart...

✓ Statistics chart saved: figures/Exp1_80_20/interactive_split_statistics_80_19.html



✓ All data split visualizations generated successfully!


## Data Split Visualization

## Run All Experiments

In [9]:
# ============================================================
# EXPERIMENT 1: Train and predict on same stock
# ============================================================
all_results = []
all_predictions = {}  # {stock: {model_type: (y_true, y_pred, dates)}}
all_histories = {}    # {stock: {model_type: history}}

for stock in STOCKS:
    print(f"\n############################################################")
    print(f"# STOCK: {stock}")
    print(f"############################################################")
    
    # Prepare data
    X_train, y_train, X_test, y_test, test_dates = prepare_same_stock_data(
        daily_data[stock], train_ratio=TRAIN_RATIO, lookback=LOOKBACK
    )
    print(f"  X_train: {X_train.shape}, X_test: {X_test.shape}")
    
    all_predictions[stock] = {}
    all_histories[stock] = {}
    
    for model_type in MODEL_TYPES:
        exp_name = f'{EXP_LABEL}_{stock}'
        
        y_true_inv, y_pred_inv, metrics, history = train_and_evaluate(
            model_type=model_type,
            X_train=X_train, y_train=y_train,
            X_test=X_test, y_test=y_test,
            experiment_name=exp_name,
            save_dir=f'models/{EXP_LABEL}',
            epochs=EPOCHS, batch_size=BATCH_SIZE
        )
        
        # Store results
        result = {'Stock': stock, 'Model': model_type, **metrics}
        all_results.append(result)
        all_predictions[stock][model_type] = (y_true_inv, y_pred_inv, test_dates)
        all_histories[stock][model_type] = history
        
        # Plot individual prediction
        plot_actual_vs_predicted(
            test_dates, y_true_inv, y_pred_inv,
            model_type, stock, EXP_LABEL,
            save_dir=f'figures/{EXP_LABEL}'
        )
        
        # Plot training history
        plot_training_history(
            history, model_type, stock, EXP_LABEL,
            save_dir=f'figures/{EXP_LABEL}'
        )

print("\n\nAll Experiment 1 (80/20) training complete!")



############################################################
# STOCK: TLKM
############################################################
  X_train: (4193, 1, 1), X_test: (1049, 1, 1)

Training BiLSTM for: Exp1_80_20_TLKM
  Train samples: 4193, Test samples: 1049
Epoch 1/100
59/59 [==============================] - ETA: 0s - loss: 0.0061
Epoch 1: val_loss improved from inf to 0.00202, saving model to models/Exp1_80_20\Exp1_80_20_TLKM_BiLSTM_best.keras
59/59 [==============================] - 14s 42ms/step - loss: 0.0061 - val_loss: 0.0020
Epoch 2/100
58/59 [============================>.] - ETA: 0s - loss: 4.2361e-04
Epoch 2: val_loss improved from 0.00202 to 0.00003, saving model to models/Exp1_80_20\Exp1_80_20_TLKM_BiLSTM_best.keras
59/59 [==============================] - 1s 15ms/step - loss: 4.1862e-04 - val_loss: 3.0447e-05
Epoch 3/100
55/59 [==========================>...] - ETA: 0s - loss: 1.0933e-04
Epoch 3: val_loss did not improve from 0.00003
59/59 [==========================

## Results Summary

In [4]:
# ============================================================
# RESULTS TABLE
# ============================================================
results_df = pd.DataFrame(all_results)
print_results_table(results_df, f"Experiment 1 - Same Stock Prediction (80/20)")

# Save results
results_df.to_csv(f'results/{EXP_LABEL}_results.csv', index=False)
print(f"Results saved to results/{EXP_LABEL}_results.csv")


NameError: name 'all_results' is not defined

## Visualizations

In [5]:
# ============================================================
# ALL MODELS COMPARISON PER STOCK
# ============================================================
for stock in STOCKS:
    y_true = all_predictions[stock][MODEL_TYPES[0]][0]
    dates = all_predictions[stock][MODEL_TYPES[0]][2]
    preds = {mt: all_predictions[stock][mt][1] for mt in MODEL_TYPES}
    
    plot_all_models_comparison(
        dates, y_true, preds, stock, EXP_LABEL,
        save_dir=f'figures/{EXP_LABEL}'
    )

print("All comparison plots saved!")


NameError: name 'all_predictions' is not defined

In [6]:
# ============================================================
# METRICS BAR CHARTS
# ============================================================
for metric in ['MSE', 'RMSE', 'MAE', 'MAPE (%)', 'R2']:
    plot_metrics_comparison_bar(
        results_df, metric, EXP_LABEL,
        group_col='Stock', save_dir=f'figures/{EXP_LABEL}'
    )

print("All metrics bar charts saved!")


  Figure saved: figures/Exp1_80_20/Exp1_80_20_MSE_comparison.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_RMSE_comparison.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_MAE_comparison.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_MAPE_pct_comparison.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_R2_comparison.png
All metrics bar charts saved!


In [6]:
# ============================================================
# SUMMARY: BEST MODEL PER STOCK
# ============================================================
print("\n" + "="*60)
print("  BEST MODEL PER STOCK (by RMSE)")
print("="*60)
for stock in STOCKS:
    stock_results = results_df[results_df['Stock'] == stock]
    best_idx = stock_results['RMSE'].idxmin()
    best = stock_results.loc[best_idx]
    print(f"  {stock}: {best['Model']} (RMSE={best['RMSE']:.4f}, R²={best['R2']:.6f})")

print("\n  BEST MODEL PER STOCK (by R² Score)")
print("="*60)
for stock in STOCKS:
    stock_results = results_df[results_df['Stock'] == stock]
    best_idx = stock_results['R2'].idxmax()
    best = stock_results.loc[best_idx]
    print(f"  {stock}: {best['Model']} (R²={best['R2']:.6f}, RMSE={best['RMSE']:.4f})")



  BEST MODEL PER STOCK (by RMSE)


NameError: name 'results_df' is not defined

## Interactive Prediction Visualization (Plotly)
Zoom in, pan, and explore the price predictions with hover details.

In [ ]:
# ============================================================
# INTERACTIVE RESULTS VISUALIZATIONS
# ============================================================

print("Generating interactive results visualizations...\n")

# 1. Interactive Results Dashboard
print("1. Generating Results Dashboard...")
fig1, html1 = create_interactive_results_dashboard_exp1(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
)
print(f"   ✓ Saved: {html1}")
fig1.show()

print()

# 2. Interactive Heatmaps for each metric
print("2. Generating Interactive Heatmaps...")
for metric in ['RMSE', 'MAE', 'R2', 'MAPE (%)']:
    try:
        fig, html = create_interactive_metrics_heatmap_exp1(
            results_df, metric, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
        )
        print(f"   ✓ {metric} heatmap: {html}")
        fig.show()
    except Exception as e:
        print(f"   ⚠ Skipping {metric}: {str(e)}")

print()

# 3. Metrics Comparison Chart
print("3. Generating Metrics Comparison Chart...")
fig3, html3 = create_interactive_metrics_comparison(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}',
    metrics=['RMSE', 'MAE', 'R2']
)
print(f"   ✓ Saved: {html3}")
fig3.show()

print()

# 4. Model Radar Chart
print("4. Generating Model Radar Chart...")
fig4, html4 = create_interactive_model_radar_chart(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
)
print(f"   ✓ Saved: {html4}")
fig4.show()

print("\n✓ All interactive results visualizations generated successfully!")

## Interactive Results Visualizations

In [8]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# INTERACTIVE VISUALIZATION - ACTUAL VS PREDICTED (Plotly)
# ============================================================

for stock in STOCKS:
    print(f"\nGenerating interactive plot for {stock}...")
    
    y_true = all_predictions[stock][MODEL_TYPES[0]][0]
    dates = all_predictions[stock][MODEL_TYPES[0]][2]
    
    # Create subplots (one for each model)
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=MODEL_TYPES,
        specs=[[{"secondary_y": False}, {"secondary_y": False}],
               [{"secondary_y": False}, {"secondary_y": False}]],
        vertical_spacing=0.12,
        horizontal_spacing=0.1
    )
    
    row_col_pairs = [(1, 1), (1, 2), (2, 1), (2, 2)]
    
    for idx, model_type in enumerate(MODEL_TYPES):
        row, col = row_col_pairs[idx]
        y_pred = all_predictions[stock][model_type][1]
        
        # Actual prices
        fig.add_trace(
            go.Scatter(
                x=dates, y=y_true,
                name='Actual Price',
                mode='lines',
                line=dict(color='blue', width=2),
                hovertemplate='<b>Actual Price</b><br>Date: %{x}<br>Price: %{y:.2f}<extra></extra>',
            ),
            row=row, col=col
        )
        
        # Predicted prices
        fig.add_trace(
            go.Scatter(
                x=dates, y=y_pred,
                name=f'{model_type} Prediction',
                mode='lines',
                line=dict(color='red', width=2, dash='dash'),
                hovertemplate='<b>Predicted Price</b><br>Date: %{x}<br>Price: %{y:.2f}<extra></extra>',
            ),
            row=row, col=col
        )
        
        # Update axes labels
        fig.update_xaxes(title_text="Date", row=row, col=col)
        fig.update_yaxes(title_text="Price", row=row, col=col)
    
    # Update layout
    fig.update_layout(
        title=f'<b>{stock} - Actual vs Predicted Prices (Experiment 1 - 80/20)</b>',
        height=900,
        template='plotly_white',
        hovermode='x unified',
        showlegend=True,
        legend=dict(
            orientation="v",
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=1.02,
        ),
        font=dict(size=10)
    )
    
    # Save as HTML
    fig.write_html(f'figures/{EXP_LABEL}/interactive_{stock}_all_models.html')
    fig.show()

print("\n✓ All interactive plots generated and saved!")
print(f"  Location: figures/{EXP_LABEL}/interactive_*.html")


Generating interactive plot for TLKM...



Generating interactive plot for BBCA...



Generating interactive plot for ASII...



Generating interactive plot for UNVR...



✓ All interactive plots generated and saved!
  Location: figures/Exp1_80_20/interactive_*.html


In [9]:
# ============================================================
# INTERACTIVE VISUALIZATION - TOGGLE MODELS WITH BUTTONS
# ============================================================

for stock in STOCKS:
    print(f"\nGenerating toggle model plot for {stock}...")
    
    y_true = all_predictions[stock][MODEL_TYPES[0]][0]
    dates = all_predictions[stock][MODEL_TYPES[0]][2]
    
    fig = go.Figure()
    
    # Add actual price (always visible)
    fig.add_trace(
        go.Scatter(
            x=dates, y=y_true,
            name='Actual Price',
            mode='lines',
            line=dict(color='blue', width=2.5),
            hovertemplate='<b>Actual Price</b><br>Date: %{x|%Y-%m-%d}<br>Price: IDR %{y:,.2f}<extra></extra>',
            visible=True
        )
    )
    
    # Add predictions for each model (togglable)
    for model_type in MODEL_TYPES:
        y_pred = all_predictions[stock][model_type][1]
        
        fig.add_trace(
            go.Scatter(
                x=dates, y=y_pred,
                name=f'{model_type} Prediction',
                mode='lines',
                line=dict(width=2),
                hovertemplate=f'<b>{model_type}</b><br>Date: %{{x|%Y-%m-%d}}<br>Price: IDR %{{y:,.2f}}<extra></extra>',
                visible=True
            )
        )
    
    # Create buttons for model selection
    buttons = [
        dict(
            label="All Models",
            method="update",
            args=[{"visible": [True] * (len(MODEL_TYPES) + 1)},
                  {"title": f"<b>{stock} - All Models vs Actual Price</b>"}]
        )
    ]
    
    for i, model_type in enumerate(MODEL_TYPES):
        visible = [True] + [False] * len(MODEL_TYPES)
        visible[i + 1] = True
        buttons.append(
            dict(
                label=model_type,
                method="update",
                args=[{"visible": visible},
                      {"title": f"<b>{stock} - {model_type} vs Actual Price</b>"}]
            )
        )
    
    # Update layout with buttons
    fig.update_layout(
        updatemenus=[
            dict(
                type="dropdown",
                direction="down",
                x=0.01,
                y=0.99,
                showactive=True,
                buttons=buttons,
                bgcolor="lightgray",
                bordercolor="gray",
                borderwidth=1,
            )
        ],
        title=f"<b>{stock} - All Models vs Actual Price</b>",
        xaxis_title="Date",
        yaxis_title="Price (IDR)",
        template="plotly_white",
        hovermode="x unified",
        height=600,
        font=dict(size=11),
        xaxis=dict(
            rangeslider=dict(visible=False),
            type="date"
        ),
        yaxis=dict(
            gridwidth=1,
            gridcolor="lightgray"
        )
    )
    
    # Save as HTML
    fig.write_html(f'figures/{EXP_LABEL}/interactive_{stock}_toggle.html')
    fig.show()

print("\n✓ All toggle model plots generated and saved!")
print(f"  Location: figures/{EXP_LABEL}/interactive_*_toggle.html")


Generating toggle model plot for TLKM...



Generating toggle model plot for BBCA...



Generating toggle model plot for ASII...



Generating toggle model plot for UNVR...



✓ All toggle model plots generated and saved!
  Location: figures/Exp1_80_20/interactive_*_toggle.html
